# DSL examples with LangChain

Anton Antonov  
[PythonForPrediction at WordPress](https://pythonforprediction.wordpress.com)  
February 2026

----

## Introduction

This notebook demonstrates the usage of the Python data package ["DSLExamples"](https://pypi.org/project/DSLExamples), [AAp1], with examples of Domain Specific Language (DSL) commands translations to programming code.

The provided DSL examples are suitable for [LLM few-shot training](https://www.prompthub.us/blog/the-few-shot-prompting-guide). [LangChain](https://www.langchain.com) can be used to create translation pipelines utilizing those examples. The utilization of such LLM-translation pipelines is exemplified below.

The Python package closely follows the Raku package  ["DSL::Examples"](https://raku.land/zef:antononcube/DSL::Examples), [AAp2], and Wolfram Language paclet ["DSLExamples"](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/DSLExamples), [AAp3], and has (or should have) the same DSL examples data.


**Remark:** Similar translations -- with much less computational resources -- are achieved with grammar-based DSL translators; see ["DSL::Translators"](https://github.com/antononcube/Raku-DSL-Translators), [AAp4].


---

## Setup


Load the packages used below:

In [22]:
from DSLExamples import dsl_examples, dsl_workflow_separators
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

import pandas as pd
import os

---

## Retrieval

Get all examples and retrieve specific language/workflow slices.


In [23]:
from_lang = 'Russian'
all_examples = dsl_examples()
python_lsa = dsl_examples("Python", "LSAMon", from_lang=from_lang)
separators = dsl_workflow_separators("WL", "LSAMon")

list(all_examples.keys()), list(python_lsa.keys())[:5]


(['WL', 'Python', 'R', 'Raku'],
 ['создать матрицу документ-терм без стемминга',
  'создать матрицу документ-терм',
  'применить функции весов терминов',
  'Вывести таблицу тем',
  'извлечь 24 темы методом NNMF, максимум шагов 12 и минимум документов на термин 10'])

---

## Tabulate Languages and Workflows


In [24]:
rows = [
    {"language": lang, "workflow": workflow}
    for lang, workflows in all_examples.items()
    for workflow in workflows.keys()
]

pd.DataFrame(rows).sort_values(["language", "workflow"]).reset_index(drop=True)


,language,workflow
0,Python,LSAMon
1,Python,QRMon
2,Python,SMRMon
3,Python,pandas
4,R,DataReshaping
5,R,LSAMon
6,R,QRMon
7,R,SMRMon
8,Raku,DataReshaping
9,Raku,SMRMon


---

## Python LSA Examples


In [25]:
pd.DataFrame([{"command": k, "code": v} for k, v in python_lsa.items()])


,command,code
0,создать матрицу документ-терм без стемминга,make_document_term_matrix(stemming_rules=False...
1,создать матрицу документ-терм,make_document_term_matrix()
2,применить функции весов терминов,apply_term_weight_functions()
3,Вывести таблицу тем,echo_topics_interpretation(wide_form=True)
4,"извлечь 24 темы методом NNMF, максимум шагов 1...","extract_topics(number_of_topics=24, method='NN..."
5,создать матрицу документ-терм с автоматическим...,"make_document_term_matrix(stemming_rules=None,..."
6,вывести таблицу тем с 10 терминами на тему,"echo_topics_interpretation(number_of_terms=10,..."
7,показать темы,echo_topics_interpretation(wide_form=True)
8,использовать dfTemp,LatentSemanticAnalyzer(dfTemp)
9,использовать документы aDocs,LatentSemanticAnalyzer(aDocs)


---

## LangChain few-shot prompt


Build a few-shot prompt from the DSL examples, then run it over commands.


In [26]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Use a small subset of examples as few-shot demonstrations
example_pairs = list(python_lsa.items())[:5]
examples = [
    {"command": cmd, "code": code}
    for cmd, code in example_pairs
]

example_prompt = PromptTemplate(
    input_variables=["command", "code"],
    template="Command: {command}\nCode: {code}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=(
        "You translate DSL commands into Python code that builds an LSA pipeline."
        "Follow the style of the examples."
    ),
    suffix="Command: {command}\nCode:",
    input_variables=["command"],
)

print(few_shot_prompt.format(command="show the topics"))


You translate DSL commands into Python code that builds an LSA pipeline.Follow the style of the examples.

Command: создать матрицу документ-терм без стемминга
Code: make_document_term_matrix(stemming_rules=False,stopWords=True)

Command: создать матрицу документ-терм
Code: make_document_term_matrix()

Command: применить функции весов терминов
Code: apply_term_weight_functions()

Command: Вывести таблицу тем
Code: echo_topics_interpretation(wide_form=True)

Command: извлечь 24 темы методом NNMF, максимум шагов 12 и минимум документов на термин 10
Code: extract_topics(number_of_topics=24, method='NNMF', max_steps=12, min_number_of_documents_per_term=10)

Command: show the topics
Code:


---

## Translation With Ollama Model

Run the few-shot prompt against a local Ollama model. 

In [27]:
llm = ChatOllama(model=os.getenv("OLLAMA_MODEL", "gemma3:12b"))

if from_lang.lower() == 'russian':
    commands = [
        "использовать aAbstracts",
        "сделать матрицу документ-терм без стемминга",
        "извлечь сорок темы методом не-негативны маричны факаторизации",
        "покажи найдёных тем",
    ]
else:
    commands = [
        "use the dataset aAbstracts",
        "make the document-term matrix without stemming",
        "extract 40 topics using the method non-negative matrix factorization",
        "show the topics",
    ]

llm = ChatOllama(model="gemma3:12b")

chain = few_shot_prompt | llm | StrOutputParser()

sep = dsl_workflow_separators('Python', 'LSAMon')

result = []
for command in commands:
    result.append(chain.invoke({"command": command}))

print(sep.join([x.strip() for x in result]))


Command: использовать aAbstracts
Code: use_data(data_source='aAbstracts')
.make_document_term_matrix(stemming_rules=False,stopWords=True)
.extract_topics(number_of_topics=40, method='NNMF')
.show_topics()


---

## Simulated Translation With a Fake LLM

For testing purposes it might be useful to use a fake LLM so the notebook runs without setup and API keys.

In [28]:
try:
    from langchain_community.llms.fake import FakeListLLM
except Exception:
    from langchain_core.language_models.fake import FakeListLLM

commands = [
    "use the dataset aAbstracts",
    "make the document-term matrix without stemming",
    "extract 40 topics using the method non-negative matrix factorization",
    "show the topics",
]

# Fake responses to demonstrate the flow
fake_responses = [
    "lsamon = lsamon_use_dataset(\"aAbstracts\")",
    "lsamon = lsamon_make_document_term_matrix(stemming=False)",
    "lsamon = lsamon_extract_topics(method=\"NMF\", n_topics=40)",
    "lsamon_show_topics(lsamon)",
]

llm = FakeListLLM(responses=fake_responses)

# Create a simple chain by piping the prompt into the LLM
chain = few_shot_prompt | llm

for command in commands:
    result = chain.invoke({"command": command})
    print("Command:", command)
    print("Code:", result)
    print("-")


Command: use the dataset aAbstracts
Code: lsamon = lsamon_use_dataset("aAbstracts")
-
Command: make the document-term matrix without stemming
Code: lsamon = lsamon_make_document_term_matrix(stemming=False)
-
Command: extract 40 topics using the method non-negative matrix factorization
Code: lsamon = lsamon_extract_topics(method="NMF", n_topics=40)
-
Command: show the topics
Code: lsamon_show_topics(lsamon)
-


---

## References

[AAp1] Anton Antonov, [DSLExamples, Python package](https://github.com/antononcube/Python-DSLExamples), (2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp2] Anton Antonov, [DSL::Examples, Raku package](https://github.com/antononcube/Raku-DSL-Examples), (2025-2026), [GitHub/antononcube](https://github.com/antononcube).

[AAp3] Anton Antonov [DSLExamples, Wolfram Language paclet](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/DSLExamples/), (2025-2026), [Wolfram Language Paclet Repository](https://resources.wolframcloud.com/publishers/resources?PublisherID=AntonAntonov).

[AAp4] Anton Antonov, [DSL::Translators, Raku package](https://github.com/antononcube/Raku-DSL-Translators), (2020-2024), [GitHub/antononcube](https://github.com/antononcube).